# EDA v2 — iFood coupons

Initial exploration of the iFood case with a deliberate separation of concerns: Spark loads and reduces data; pandas analyzes, tabulates, and prepares visualizations.

```mermaid
flowchart LR
  subgraph spark [Spark]
    RawJSON[raw JSONs]
    Processed[processed parquet]
    Agg[groupBy / count / filter]
    RawJSON --> Agg
    Processed --> Agg
  end
  subgraph pandas [Pandas]
    Small[Small DataFrames]
    Tables[tables and stats]
    Plots[Plotly figures]
    Small --> Tables
    Small --> Plots
  end
  Agg -->|toPandas| Small
```

Figures use primitives and theme from `src/viz.py` (statistical-plots skill); the notebook only aggregates and displays.

## 1. Setup

Source: raw and processed. Initializes paths, configuration, and Spark DataFrames, leaving load and cache to Spark.

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.clean import normalize_profile
from src.config import load
from src.io import parse_events, read_offers, read_profile
from src.pipeline import build_spark

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

cfg = load()
spark = build_spark(cfg, app_name="eda-v2")
spark.sparkContext.setLogLevel("ERROR")

events = parse_events(spark, cfg).cache()
offers = read_offers(spark, cfg).cache()
raw_profile = read_profile(spark, cfg).cache()
profile = normalize_profile(raw_profile, cfg).cache()
processed = spark.read.parquet(str(cfg.processed_dir)).cache()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/15 16:26:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Data overview

Source: raw and processed. Measures source and analytical-grain volume before interpreting any rate.

In [2]:
from pyspark.sql import functions as F

from src import viz
from src.viz import (
    format_pt_br,
    horizontal_bars,
    setup_theme,
    vertical_share_bars,
)

# SPARK REDUCE
n_linhas = processed.count()
n_clientes = processed.select("account_id").distinct().count()
n_ofertas_processed = processed.select("offer_id").distinct().count()
n_events = events.count()

eventos_por_tipo = (
    events.groupBy("event")
    .agg(F.count("*").alias("eventos"), F.min("time").alias("primeiro_dia"))
    .toPandas()
    .sort_values("eventos", ascending=False)
    .reset_index(drop=True)
)

volumetria = pd.DataFrame([
    {
        "fonte": "events (raw)",
        "linhas": n_events,
        "clientes": events.select("account_id").distinct().count(),
        "ofertas": pd.NA,
        "grao": "4 event types",
    },
    {
        "fonte": "profile (raw)",
        "linhas": profile.count(),
        "clientes": profile.select("account_id").distinct().count(),
        "ofertas": pd.NA,
        "grao": "1 linha por cliente",
    },
    {
        "fonte": "offers (raw)",
        "linhas": offers.count(),
        "clientes": pd.NA,
        "ofertas": offers.select("id").distinct().count(),
        "grao": "offer catalog",
    },
    {
        "fonte": "processed",
        "linhas": n_linhas,
        "clientes": n_clientes,
        "ofertas": n_ofertas_processed,
        "grao": "(account_id, offer_id, received_time)",
    },
])

# PANDAS ANALYSIS
eventos_por_tipo["fracao"] = eventos_por_tipo["eventos"] / eventos_por_tipo["eventos"].sum()

display(volumetria)
display(eventos_por_tipo)

ratio = n_linhas / n_events
print(
    f"The processed grain condenses {format_pt_br(n_events)} raw events into "
    f"{format_pt_br(n_linhas)} receipts ({ratio:.1%} of event volume), "
    f"covering {format_pt_br(n_clientes)} costmers and the {n_ofertas_processed} catalog offers."
)

setup_theme()

horizontal_bars(
    volumetria,
    category="fonte",
    value="linhas",
    title="Raw data is large; the processed grain condenses what matters",
    subtitle=f"events: {format_pt_br(n_events)} · processed: {format_pt_br(n_linhas)} · log scale",
    xlabel="Linhas",
    log_scale=False,
).show()

horizontal_bars(
    volumetria.dropna(subset=["clientes"]),
    category="fonte",
    value="clientes",
    title="Nearly all costmers appear across the three sources",
    subtitle=f"Processed grain covers {format_pt_br(n_clientes)} clientes",
    xlabel="Clientes",
    color=viz.palette()[1],
).show()

vertical_share_bars(
    eventos_por_tipo,
    category="event",
    value="eventos",
    share="fracao",
    title="Transactions dominate the raw event stream",
    subtitle=f"n = {format_pt_br(n_events)} eventos",
).show()

,fonte,linhas,clientes,ofertas,grao
0,events (raw),306534,17000,<NA>,4 event types
1,profile (raw),17000,17000,<NA>,1 linha por cliente
2,offers (raw),10,<NA>,10,offer catalog
3,processed,76277,16994,10,"(account_id, offer_id, received_time)"


,event,eventos,primeiro_dia,fracao
0,transaction,138953,0.0,0.453304
1,offer received,76277,0.0,0.248837
2,offer viewed,57725,0.0,0.188315
3,offer completed,33579,0.0,0.109544


The processed grain condenses 306.534 raw events into 76.277 receipts (24.9% of event volume), covering 16.994 costmers and the 10 catalog offers.


## 3. Offer catalog

Source: raw. Summarizes types, discounts, minimums, duration, and channels of available offers to understand the decision space.

In [3]:
from src import viz
from src.viz import format_pt_br, grouped_bars, horizontal_bars

# SPARK REDUCE
catalogo = (
    offers.select("id", "offer_type", "min_value", "duration", "discount_value", "channels")
    .toPandas()
    .sort_values(["offer_type", "discount_value", "min_value"])
    .reset_index(drop=True)
)

por_tipo = (
    catalogo.groupby("offer_type", as_index=False)
    .agg(
        ofertas=("id", "count"),
        min_medio=("min_value", "mean"),
        desconto_medio=("discount_value", "mean"),
        duracao_media=("duration", "mean"),
    )
)

# PANDAS ANALYSIS
catalogo_plot = catalogo.assign(
    rotulo=lambda d: (
        d["offer_type"].str[:4]
        + " · desc "
        + d["discount_value"].astype(int).astype(str)
        + " · min "
        + d["min_value"].astype(int).astype(str)
    ),
)

display(catalogo)
display(por_tipo.round(1))

print(
    f"{len(catalogo)} offers, {catalogo['offer_type'].nunique()} tipos — "
    f"informational has zero discount and minimum; duration ranges from "
    f"{int(catalogo['duration'].min())} a {int(catalogo['duration'].max())} days."
)

horizontal_bars(
    por_tipo,
    category="offer_type",
    value="ofertas",
    title="The catalog is small and imbalanced by type",
    subtitle="4 bogo · 4 discount · 2 informational",
    xlabel="Offers in catalog",
    color=viz.palette()[0],
).show()

grouped_bars(
    catalogo_plot.sort_values(["offer_type", "discount_value"]),
    category="rotulo",
    series=["min_value", "discount_value", "duration"],
    series_names=["Minimum spend", "Discount", "Duration (days)"],
    orientation="h",
    title="Each offer combines its own discount rule, minimum, and validity window",
    subtitle="informational: zero reward · validity is not constant in the pipeline",
    xlabel="Value / days",
    value_label="%{x:.0f}",
).show()

,id,offer_type,min_value,duration,discount_value,channels
0,9b98b8c7a33c4b65b9aebfe6a799e6d9,bogo,5,7.0,5,"[web, email, mobile]"
1,f19421c1d4aa40978ebb69ca19b0e20d,bogo,5,5.0,5,"[web, email, mobile, social]"
2,ae264e3637204a6fb9bb56bc8210ddfd,bogo,10,7.0,10,"[email, mobile, social]"
3,4d5c57ea9a6940dd891ad53e9dbe8da0,bogo,10,5.0,10,"[web, email, mobile, social]"
4,fafdcd668e3743c1bb461111dcafc2a4,discount,10,10.0,2,"[web, email, mobile, social]"
5,2906b810c7d4411798c6938adc9daaa5,discount,10,7.0,2,"[web, email, mobile]"
6,2298d6c36e964ae4a3e7e9706d1fb8c2,discount,7,7.0,3,"[web, email, mobile, social]"
7,0b1e1539f2cc45b7b9fa7c272da2e1d7,discount,20,10.0,5,"[web, email]"
8,3f207df678b143eea3cee63160fa8bed,informational,0,4.0,0,"[web, email, mobile]"
9,5a8bc65990b245e5a138643cd4eb9837,informational,0,3.0,0,"[email, mobile, social]"


,offer_type,ofertas,min_medio,desconto_medio,duracao_media
0,bogo,4,7.5,7.5,6.0
1,discount,4,11.8,3.0,8.5
2,informational,2,0.0,0.0,3.5


10 offers, 3 tipos — informational has zero discount and minimum; duration ranges from 3 a 10 days.


## 4. Events over time

Source: raw. Checks when each event type occurs and whether sends are discrete waves or continuous activity.

In [4]:
from src import eda
from src.viz import faceted, format_pt_br, timeline_ranges

# SPARK REDUCE
por_dia = eda.events_over_time(events)
ondas = sorted(por_dia.loc[por_dia["event"] == "offer received", "dia"].unique())
dia_max = int(por_dia["dia"].max())
janelas, fim_do_teste = eda.campaign_validity_windows(processed, events)

totais = (
    por_dia.groupby("event", observed=False)["eventos"]
    .sum()
    .reindex(eda.EVENT_ORDER)
    .reset_index()
)
n_censurados = int(janelas["recebimentos_censurados"].sum())
n_linhas = processed.count()

# PANDAS ANALYSIS
display(totais)
display(janelas.round(3))
print(
    f"Sends on {len(ondas)} days (t={', '.join(str(int(d)) for d in ondas)}); "
    f"end of data at t={fim_do_teste:g}."
)

rotulos = {
    "offer received": "receipts per day",
    "offer viewed": "views per day",
    "offer completed": "completions per day",
    "transaction": "purchases per day",
}
paineis = []
for evento in eda.EVENT_ORDER:
    sub = por_dia.loc[por_dia["event"] == evento].sort_values("dia")
    paineis.append({
        "title": rotulos[evento],
        "x": sub["dia"].tolist(),
        "y": sub["eventos"].tolist(),
        "kind": "line" if evento == "transaction" else "bar",
    })

faceted(
    paineis,
    title="Six campaign sends; response fades across waves",
    subtitle=(
        f"Waves at t={', '.join(str(int(d)) for d in ondas)} · "
        "linear scale per panel · dashed = send"
    ),
    cols=2,
    kind="bar",
    vlines=ondas,
    xlabel="day since test start",
    x_range=(-0.5, dia_max + 0.5),
    x_dtick=5,
    y_tickformat=",.0f",
    row_height=220,
).show()

timeline_ranges(
    janelas,
    label="rotulo",
    start="received_time",
    end="valid_until",
    observed_end="janela_observavel",
    censored="censurada",
    title="Late waves have validity cut off by the end of data collection",
    subtitle=(
        f"{format_pt_br(n_censurados)} receipts ({n_censurados / n_linhas:.1%}) "
        "with window beyond t=" + f"{fim_do_teste:g}"
    ),
    xlabel="day since test start",
    vline=fim_do_teste,
).show()

print(
    f"Right censoring: {format_pt_br(n_censurados)} recebimentos "
    f"({n_censurados / n_linhas:.1%}) do not observe the full validity — "
    "conversion in waves 4–5 is underestimated."
)

,event,eventos
0,offer received,76277
1,offer viewed,57725
2,offer completed,33579
3,transaction,138953


,campaign_wave,received_time,duration,recebimentos,valid_until,janela_observavel,censurada,dias_censurados,rotulo,recebimentos_censurados
0,0,0.0,6.526,12650,6.526,6.526,False,0.000,wave 0 · t=0 · 6.52569d,0
1,1,7.0,6.495,12669,13.495,13.495,False,0.000,wave 1 · t=7 · 6.49491d,0
2,2,14.0,6.511,12711,20.511,20.511,False,0.000,wave 2 · t=14 · 6.51066d,0
3,3,17.0,6.480,12778,23.480,23.480,False,0.000,wave 3 · t=17 · 6.48036d,0
4,4,21.0,6.508,12704,27.508,27.508,False,0.000,wave 4 · t=21 · 6.50834d,0
5,5,24.0,6.502,12765,30.502,29.750,True,0.752,wave 5 · t=24 · 6.50247d,12765


Sends on 6 days (t=0, 7, 14, 17, 21, 24); end of data at t=29.75.


Right censoring: 12.765 recebimentos (16.7%) do not observe the full validity — conversion in waves 4–5 is underestimated.


## 5. Raw data quality

Source: raw. Looks for visible problems before the pipeline: missing references, out-of-order events, missing profile, and sentinels.

In [5]:
from pyspark.sql import functions as F

from src import eda
from src.attribution import attribute
from src.viz import format_pt_br, vertical_bars

# SPARK REDUCE
n_txn = events.filter(F.col("event") == "transaction").count()
n_txn_ref = events.filter(
    (F.col("event") == "transaction") & F.col("offer_ref").isNotNull()
).count()
n_camp = events.filter(F.col("event") != "transaction").count()
n_camp_ref = events.filter(
    (F.col("event") != "transaction") & F.col("offer_ref").isNotNull()
).count()
dup_events = (
    events.groupBy("account_id", "event", "time", "offer_ref")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

sem_view = eda.completed_unseen_by_type(events, offers)
nulos_perfil = eda.identity_null_overlap(raw_profile, cfg)
attributed = attribute(events, offers, cfg).cache()
fora_janela = eda.unattributable_transaction_share(events, attributed)
pago_abaixo = eda.paid_below_minimum(processed)

com_evento = sem_view[sem_view["completados"] > 0]
taxa_sem_view = com_evento["sem_view"].sum() / com_evento["completados"].sum()

# PANDAS ANALYSIS
qualidade = pd.DataFrame([
    {
        "metric": "transactions without offer_ref",
        "numerador": n_txn - n_txn_ref,
        "denominador": n_txn,
        "taxa": (n_txn - n_txn_ref) / n_txn,
        "reading": "expected — offer→purchase link is via temporal attribution",
    },
    {
        "metric": "campaign events without offer_ref",
        "numerador": n_camp - n_camp_ref,
        "denominador": n_camp,
        "taxa": (n_camp - n_camp_ref) / n_camp,
        "reading": "problem if > 0",
    },
    {
        "metric": "offer completed without prior view",
        "numerador": int(com_evento["sem_view"].sum()),
        "denominador": int(com_evento["completados"].sum()),
        "taxa": taxa_sem_view,
        "reading": "expected — control converts without exposure (G3)",
    },
    {
        "metric": "transactions outside any offer window",
        "numerador": int(fora_janela.loc[fora_janela["grupo"] == "outside any offer window", "transacoes"].iloc[0]),
        "denominador": int(fora_janela["transacoes"].sum()),
        "taxa": float(fora_janela.loc[fora_janela["grupo"] == "outside any offer window", "fracao"].iloc[0]),
        "reading": "spontaneous behavior — not attributable to send",
    },
    {
        "metric": "profile with age sentinel (118)",
        "numerador": int(nulos_perfil.loc[nulos_perfil["conjunto"] == f"age={cfg.age_sentinel}", "clientes"].iloc[0]),
        "denominador": int(nulos_perfil["total_clientes"].iloc[0]),
        "taxa": float(nulos_perfil.loc[nulos_perfil["conjunto"] == f"age={cfg.age_sentinel}", "fracao"].iloc[0]),
        "reading": "expected — becomes identity_missing in the pipeline",
    },
    {
        "metric": "duplicate raw rows (account, event, time, offer_ref)",
        "numerador": dup_events,
        "denominador": events.count(),
        "taxa": dup_events / events.count(),
        "reading": "problem if > 0",
    },
])

display(qualidade.assign(taxa=lambda d: d["taxa"].map("{:.1%}".format)))
display(nulos_perfil)
display(pago_abaixo)

vertical_bars(
    sem_view[sem_view["completados"] > 0],
    category="offer_type",
    value="taxa_sem_view",
    title="A quarter of completions were never viewed — real control in the funnel",
    subtitle=f"overall rate {taxa_sem_view:.1%} on completions",
    ylabel="Rate without prior view",
    label_template="%{y:.1%}",
    tickformat=".0%",
    hovertemplate="%{x}<br>%{y:.1%}<extra></extra>",
).show()

print(
    f"Transactions with offer_ref: {n_txn_ref} of {format_pt_br(n_txn)} — "
    "none; the offer→purchase link exists only through temporal attribution."
)
print(
    f"Paid conversions below minimum (G10 audit): "
    f"{int(pago_abaixo['abaixo_do_minimo'].sum())} — must be zero."
)

Premissa 1 violada em 15721 transação(ões): mais de uma oferta ativa no intervalo; prioridade 'earliest_received' aplicada.


,metric,numerador,denominador,taxa,reading
0,transactions without offer_ref,138953,138953,100.0%,expected — offer→purchase link is via temporal...
1,campaign events without offer_ref,0,167581,0.0%,problem if > 0
2,offer completed without prior view,8568,33182,25.8%,expected — control converts without exposure (G3)
3,transactions outside any offer window,17090,138953,12.3%,spontaneous behavior — not attributable to send
4,profile with age sentinel (118),2175,17000,12.8%,expected — becomes identity_missing in the pip...
5,"duplicate raw rows (account, event, time, offe...",396,306534,0.1%,problem if > 0


,conjunto,clientes,fracao,total_clientes
0,age=118,2175,0.127941,17000
1,gender nulo,2175,0.127941,17000
2,credit_card_limit nulo,2175,0.127941,17000
3,"all three, together",2175,0.127941,17000
4,ao menos um,2175,0.127941,17000


,offer_type,conversoes_pagas,abaixo_do_minimo,cost_acima_da_revenue,cost_total,cost_sob_suspeita,frac_abaixo_do_minimo,frac_cost_sob_suspeita
0,bogo,13496,0,0,96795.0,0.0,0.0,0.0
1,discount,12500,0,0,34773.0,0.0,0.0,0.0


Transactions with offer_ref: 0 of 138.953 — none; the offer→purchase link exists only through temporal attribution.
Paid conversions below minimum (G10 audit): 0 — must be zero.


## 6. Processed grain

Source: processed. Confirms the analytical unit of the prepared dataset and flags nulls, duplicates, or columns that need care.

In [6]:
from pyspark.sql import functions as F

from src import eda
from src.contract import CONTRACT_COLUMNS, NULLABLE_COLUMNS

# SPARK REDUCE
n_linhas = processed.count()
dupes = (
    processed.groupBy("account_id", "offer_id", "received_time")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
nulos_row = processed.select(
    *[F.sum(F.col(c).isNull().cast("long")).alias(c) for c in CONTRACT_COLUMNS]
).first().asDict()
contagens = (
    processed.groupBy("offer_type", "treatment", "converted", "campaign_wave")
    .agg(F.count("*").alias("linhas"))
    .orderBy("campaign_wave", "offer_type", "treatment", "converted")
    .toPandas()
)
amostra = (
    processed.orderBy("campaign_wave", "account_id", "offer_id")
    .limit(5)
    .toPandas()
)

# PANDAS ANALYSIS
nulos = pd.DataFrame(
    [{"coluna": c, "nulos": int(nulos_row[c] or 0)} for c in CONTRACT_COLUMNS if (nulos_row[c] or 0) > 0]
)
checks = eda.sanity_checks(processed)

display(pd.DataFrame([
    {"check": "unique grain (account_id, offer_id, received_time)", "valor": dupes, "esperado": 0},
    {"check": "rows at grain", "valor": n_linhas, "esperado": "76.277"},
]))
display(nulos if len(nulos) else pd.DataFrame({"coluna": ["—"], "nulos": [0]}))
display(contagens.head(12))
display(checks)
display(amostra)

print(
    "Analytical unit: one row per offer receipt "
    "(account_id, offer_id, received_time), with label and features as-of that instant."
)

,check,valor,esperado
0,"unique grain (account_id, offer_id, received_t...",0,0
1,rows at grain,76277,76.277


,coluna,nulos
0,age,9776
1,credit_card_limit,9776
2,hist_recency_days,20952
3,hist_time_view_to_conv,47850


,offer_type,treatment,converted,campaign_wave,linhas
0,bogo,0,0,0,478
1,bogo,0,1,0,230
2,bogo,1,0,0,2264
3,bogo,1,1,0,2046
4,discount,0,0,0,1073
5,discount,0,1,0,292
6,discount,1,0,0,1851
7,discount,1,1,0,1877
8,informational,0,0,0,541
9,informational,0,1,0,318


,check,linhas,fracao
0,converted=1 com conversion_value = 0,0,0.0
1,conversion_value > 0 sem converted,0,0.0
2,reward_cost > 0 without conversion (violates G6),0,0.0
3,reward_cost > 0 em informational (viola G6),0,0.0
4,reward_cost acima da revenue da conversão,0,0.0
5,historical average ticket without historical t...,0,0.0
6,negative historical spend,0,0.0
7,"historical view rate outside [0,1]",0,0.0
8,age equals sentinel (violates G7),0,0.0


,account_id,offer_id,offer_type,received_time,campaign_wave,treatment,converted,conversion_value,reward_cost,is_recurrent,age,gender,credit_card_limit,identity_missing,tenure_days,hist_spend_total,hist_txn_count,hist_avg_ticket,hist_spend_std,hist_recency_days,hist_frequency,hist_spend_trend,hist_offers_received,hist_offers_received_bogo,hist_offers_received_discount,hist_offers_received_info,hist_offers_viewed,hist_offers_completed,hist_view_rate,hist_conv_rate_bogo,hist_conv_rate_discount,hist_completed_unseen_flag,hist_time_view_to_conv,discount_value,min_value,duration,n_channels,channel_web,channel_email,channel_mobile,channel_social,discount_to_minvalue_ratio,n_concurrent_offers
0,0011e0d4e6b944f998e987f904e8c1e5,3f207df678b143eea3cee63160fa8bed,informational,0.0,0,1,0,0.00,0.0,0,40,O,57000.0,0,198,0.0,0,0.0,0.0,NaN,0.0,0.0,0,0,0,0,0,0,0.0,0.0,0.0,0,NaN,0.0,0.0,4.0,3,1,1,1,0,0.0,0
1,0020c2b971eb4e9188eac86d93036a77,fafdcd668e3743c1bb461111dcafc2a4,discount,0.0,0,1,1,17.63,2.0,0,59,F,90000.0,0,874,0.0,0,0.0,0.0,NaN,0.0,0.0,0,0,0,0,0,0,0.0,0.0,0.0,0,NaN,2.0,10.0,10.0,4,1,1,1,1,0.2,0
2,003d66b6608740288d6cc97a6903f4f0,5a8bc65990b245e5a138643cd4eb9837,informational,0.0,0,1,1,0.44,0.0,0,26,F,73000.0,0,400,0.0,0,0.0,0.0,NaN,0.0,0.0,0,0,0,0,0,0,0.0,0.0,0.0,0,NaN,0.0,0.0,3.0,3,0,1,1,1,0.0,0
3,00426fe3ffde4c6b9cb9ad6d077a13ea,5a8bc65990b245e5a138643cd4eb9837,informational,0.0,0,1,1,5.33,0.0,0,19,F,65000.0,0,716,0.0,0,0.0,0.0,NaN,0.0,0.0,0,0,0,0,0,0,0.0,0.0,0.0,0,NaN,0.0,0.0,3.0,3,0,1,1,1,0.0,0
4,005500a7188546ff8a767329a2f7c76a,ae264e3637204a6fb9bb56bc8210ddfd,bogo,0.0,0,1,0,0.00,0.0,0,56,M,47000.0,0,229,0.0,0,0.0,0.0,NaN,0.0,0.0,0,0,0,0,0,0,0.0,0.0,0.0,0,NaN,10.0,10.0,7.0,3,0,1,1,1,1.0,0


Analytical unit: one row per offer receipt (account_id, offer_id, received_time), with label and features as-of that instant.


## 7. Customer profile and features

Source: raw and processed. Describes registration, historical features built by the pipeline (`hist_*`), and observed exposure in the test.

In [7]:
import numpy as np

from pyspark.sql import functions as F

from src import eda
from src.viz import faceted, format_pt_br, heatmap, vertical_bars_ci

# SPARK REDUCE — registration profile
perfil_num = eda.numeric_profile(profile, ["age", "credit_card_limit", "tenure_days"], cfg)
perfil_cat = eda.categorical_profile(profile, ["gender"])
clientes = eda.client_features(processed, events)

n_profile = profile.select("account_id").distinct().count()
n_processed = processed.select("account_id").distinct().count()
n_sentinela = profile.filter(F.col("identity_missing") == 1).count()

hist_compra = eda.numeric_profile(processed, list(eda.HIST_COMPRA_FEATURES), cfg)
hist_oferta = eda.numeric_profile(processed, list(eda.HIST_OFERTA_FEATURES), cfg)
corr_hist = eda.correlation_matrix(
    processed,
    ["hist_spend_total", "hist_txn_count", "hist_avg_ticket", "hist_frequency", "hist_view_rate"],
)
redundantes = eda.redundant_pairs(corr_hist, cfg)

cobertura = pd.DataFrame([
    {"fonte": "profile", "clientes": n_profile},
    {"fonte": "processed", "clientes": n_processed},
    {"fonte": "identity_missing=1", "clientes": n_sentinela},
])
exposicao = clientes[["offers_received", "offers_viewed", "view_rate", "conv_rate"]].describe()

# PANDAS ANALYSIS
display(cobertura)
display(perfil_num.round(3))
display(perfil_cat)
display(hist_compra.round(3))
display(hist_oferta.round(3))
display(redundantes if len(redundantes) else pd.DataFrame({"feature_a": [], "feature_b": [], "r": []}))
display(exposicao.round(3))

panels_perfil = []
for coluna, frame in [
    ("credit_card_limit", profile),
    ("tenure_days", profile),
    ("spend_total", None),
]:
    if frame is not None:
        hist = eda.numeric_histogram(frame, coluna, cfg.histogram_bins)
    else:
        hist = (
            clientes[["spend_total"]]
            .assign(bucket=pd.cut(clientes["spend_total"], bins=cfg.histogram_bins))
            .groupby("bucket", observed=False)
            .size()
            .reset_index(name="contagem")
        )
        hist["centro"] = hist["bucket"].apply(lambda b: b.mid if pd.notna(b) else np.nan)
        hist = hist.dropna(subset=["centro"])
    panels_perfil.append({"title": coluna, "x": hist["centro"].tolist(), "y": hist["contagem"].tolist()})

faceted(
    panels_perfil,
    title="Registration and spend observed in the test",
    subtitle=f"{format_pt_br(n_profile)} clientes · {format_pt_br(n_sentinela)} with missing identity",
    kind="bar",
    cols=3,
    xlabel="valor",
).show()

panels_hist = []
for coluna in ["hist_spend_total", "hist_view_rate", "hist_offers_received_discount"]:
    hist = eda.numeric_histogram(processed, coluna, cfg.histogram_bins)
    panels_hist.append({"title": coluna, "x": hist["centro"].tolist(), "y": hist["contagem"].tolist()})

faceted(
    panels_hist,
    title="Historical features carry spend tails and response habits",
    subtitle="computed as-of received_time · legitimate zeros where there is no history",
    kind="bar",
    cols=3,
    xlabel="valor",
).show()

heatmap(
    corr_hist,
    title="Purchase history and view rate are not orthogonal",
    subtitle=f"|r| ≥ {cfg.correlation_threshold} signals redundancy",
    diverging=True,
).show()

ticket_grupos = eda.conversion_group_ticket(processed, cfg)
display(ticket_grupos.round(2))
vertical_bars_ci(
    ticket_grupos,
    category="grupo",
    value="ticket_medio",
    value_lo="ticket_lo",
    value_hi="ticket_hi",
    title="All observed ticket sits on converters — non-converters add zero revenue",
    subtitle=(
        "ticket_medio = mean(conversion_value) per receipt · "
        f"IC {cfg.gain_curve_confidence_level:.0%} bootstrap ({cfg.gain_curve_n_bootstrap} réplicas)"
    ),
    ylabel="R$ por envio",
).show()

print(
    f"Processed covers {format_pt_br(n_processed)} de {format_pt_br(n_profile)} clientes; "
    f"as `hist_*` summarize pre-send behavior per receipt row."
)

,fonte,clientes
0,profile,17000
1,processed,16994
2,identity_missing=1,2175


,coluna,n,nulos,frac_nulos,zeros,frac_zeros,media,desvio,min,p1,p25,p50,p75,p99,max,outliers,frac_outliers
0,age,14825,2175,0.128,0,0.000,54.394,17.384,18.0,19.0,42.0,55.0,66.0,92.0,101.0,0,0.000
1,credit_card_limit,14825,2175,0.128,0,0.000,65404.992,21598.299,30000.0,31000.0,49000.0,64000.0,80000.0,116000.0,120000.0,0,0.000
2,tenure_days,17000,0,0.000,22,0.001,517.450,411.224,0.0,8.0,208.0,358.0,789.0,1727.0,1823.0,295,0.017


,nivel,linhas,coluna,fracao
0,M,8484,gender,0.499059
1,F,6129,gender,0.360529
2,unknown,2175,gender,0.127941
3,O,212,gender,0.012471


,coluna,n,nulos,frac_nulos,zeros,frac_zeros,media,desvio,min,p1,p25,p50,p75,p99,max,outliers,frac_outliers
0,hist_spend_total,76277,0,0.000,20952,0.275,43.816,78.499,0.00,0.000,0.000,16.140,59.750,276.260,1325.720,4542,0.060
1,hist_txn_count,76277,0,0.000,20952,0.275,3.457,3.667,0.00,0.000,0.000,3.000,5.000,15.000,29.000,2214,0.029
2,hist_avg_ticket,76277,0,0.000,20952,0.275,9.763,17.992,0.00,0.000,0.000,3.780,16.720,37.350,962.100,653,0.009
3,hist_spend_std,76277,0,0.000,29062,0.381,4.436,22.427,0.00,0.000,0.000,1.610,4.298,37.223,679.657,2323,0.030
4,hist_recency_days,55325,20952,0.275,0,0.000,3.354,3.083,0.25,0.250,1.000,2.500,4.750,13.750,24.000,2234,0.040
5,hist_frequency,76277,0,0.000,20952,0.275,0.203,0.202,0.00,0.000,0.000,0.143,0.294,0.857,2.000,1412,0.019
6,hist_spend_trend,76277,0,0.000,20961,0.275,0.885,24.074,-962.10,-29.915,-1.212,0.000,2.196,31.030,961.180,16247,0.213


,coluna,n,nulos,frac_nulos,zeros,frac_zeros,media,desvio,min,p1,p25,p50,p75,p99,max,outliers,frac_outliers
0,hist_offers_received,76277,0,0.000,16994,0.223,1.872,1.454,0.0,0.0,1.000,2.00,3.000,5.0,5.00,0,0.000
1,hist_offers_viewed,76277,0,0.000,21991,0.288,1.425,1.252,0.0,0.0,0.000,1.00,2.000,5.0,5.00,0,0.000
2,hist_offers_completed,76277,0,0.000,43057,0.564,0.738,1.028,0.0,0.0,0.000,0.00,1.000,4.0,5.00,5871,0.077
3,hist_offers_received_bogo,76277,0,0.000,37434,0.491,0.747,0.886,0.0,0.0,0.000,1.00,1.000,3.0,5.00,3474,0.046
4,hist_offers_received_discount,76277,0,0.000,37356,0.490,0.751,0.891,0.0,0.0,0.000,1.00,1.000,3.0,5.00,3509,0.046
5,hist_view_rate,76277,0,0.000,21991,0.288,0.598,0.426,0.0,0.0,0.000,0.75,1.000,1.0,1.00,0,0.000
6,hist_conv_rate_bogo,76277,0,0.000,56006,0.734,0.238,0.409,0.0,0.0,0.000,0.00,0.500,1.0,1.00,0,0.000
7,hist_conv_rate_discount,76277,0,0.000,54131,0.710,0.261,0.423,0.0,0.0,0.000,0.00,0.500,1.0,1.00,0,0.000
8,hist_completed_unseen_flag,76277,0,0.000,64682,0.848,0.152,0.359,0.0,0.0,0.000,0.00,0.000,1.0,1.00,11595,0.152
9,hist_time_view_to_conv,28427,47850,0.627,2063,0.073,2.003,1.872,0.0,0.0,0.625,1.50,2.875,8.0,22.25,901,0.032


,feature_a,feature_b,r,abs_r
0,hist_txn_count,hist_frequency,0.881902,0.881902


,offers_received,offers_viewed,view_rate,conv_rate
count,16994.000,16994.000,16994.00,16994.000
mean,4.488,3.216,0.72,0.449
std,1.073,1.273,0.24,0.319
min,1.000,0.000,0.00,0.000
25%,4.000,2.000,0.50,0.200
50%,5.000,3.000,0.75,0.500
75%,5.000,4.000,1.00,0.667
max,6.000,6.000,1.00,1.000


,grupo,received,ticket_medio,ticket_lo,ticket_hi
0,did not convert,42273,0.00,0.00,0.00
1,converted,34004,19.49,19.08,19.89


Processed covers 16.994 de 17.000 clientes; as `hist_*` summarize pre-send behavior per receipt row.


## 8. Funnel, conversion, and repurchase recurrence

Source: processed. Measures received, viewed, converted, and repurchase within the recurrence window, with explicit denominators.

In [8]:
from pyspark.sql import functions as F

import numpy as np

from src import eda
from src.viz import format_pt_br, grouped_bars, line_series, vertical_bars

# SPARK REDUCE
funil = eda.response_funnel(processed)
ondas = eda.campaign_waves(processed)
por_wave = (
    processed.groupBy("campaign_wave")
    .agg(
        F.count("*").alias("received"),
        F.sum("treatment").alias("viewed"),
        F.sum("converted").alias("converted"),
    )
    .orderBy("campaign_wave")
    .toPandas()
    .assign(
        taxa_view=lambda d: d["viewed"] / d["received"],
        taxa_conversao=lambda d: d["converted"] / d["received"],
        taxa_conversao_viewed=lambda d: np.where(d["viewed"] > 0, d["converted"] / d["viewed"], np.nan),
    )
)
rec_wave = eda.recurrence_by_wave(processed)
rec_trat = eda.recurrence_by_treatment(processed)
rec_pivot = (
    rec_trat.assign(grupo=rec_trat["treatment"].map({0: "did not view", 1: "viewed"}))
    .pivot(index="offer_type", columns="grupo", values="taxa_recorrencia_converted")
    .reset_index()
)

# PANDAS ANALYSIS
display(funil.round(4))
display(por_wave.round(4))
display(rec_wave.round(4))
display(rec_trat.round(4))

taxa_media_rec = funil["taxa_recorrencia_converted"].mean()

grouped_bars(
    funil,
    category="offer_type",
    series=["received", "viewed", "converted", "recorrentes"],
    series_names=["received", "viewed", "converted", f"recorrentes ({cfg.recurrence_window_days}d)"],
    title="Full funnel by type — received, viewed, converted, and recurrent",
    subtitle=f"recurrent = another purchase within {cfg.recurrence_window_days} days after conversion",
    ylabel="linhas",
    orientation="v",
).show()

vertical_bars(
    funil,
    category="offer_type",
    value="taxa_conversao",
    title="Conversion rate on received differs from rate on viewed",
    subtitle="taxa_conversao = converted / received",
    ylabel="Rate on received",
    label_template="%{y:.1%}",
    tickformat=".0%",
    hovertemplate="%{x}<br>%{y:.1%}<extra></extra>",
).show()

vertical_bars(
    funil,
    category="offer_type",
    value="taxa_recorrencia_converted",
    title=f"{taxa_media_rec:.0%} of converters, on average, repurchase within {cfg.recurrence_window_days} days",
    subtitle="taxa_recorrencia_converted = recurrent / converted",
    ylabel="Rate on converted",
    label_template="%{y:.1%}",
    tickformat=".0%",
    hovertemplate="%{x}<br>%{y:.1%}<extra></extra>",
).show()

line_series(
    por_wave,
    x="campaign_wave",
    y="taxa_conversao",
    title="Conversion drops in late waves — right censoring",
    subtitle="taxa_conversao = converted / received por onda",
    xlabel="campaign wave",
    ylabel="conversion rate",
    tickformat=".0%",
    mode="lines+markers",
).show()

line_series(
    rec_wave,
    x="campaign_wave",
    y="taxa_recorrencia_converted",
    title="Recurrence among converters is stable across waves",
    subtitle=f"taxa_recorrencia_converted = recurrent / converted · janela {cfg.recurrence_window_days} dias",
    xlabel="campaign wave",
    ylabel="recurrence rate",
    tickformat=".0%",
    mode="lines+markers",
).show()

grouped_bars(
    rec_pivot,
    category="offer_type",
    series=["did not view", "viewed"],
    title="Recurrence among converters is similar with and without view",
    subtitle=f"taxa_recorrencia_converted by arm · window {cfg.recurrence_window_days} dias",
    ylabel="rate on converted",
    orientation="v",
    value_label="%{y:.1%}",
).update_layout(yaxis_tickformat=".0%").show()

total_rec = int(funil["recorrentes"].sum())
total_conv = int(funil["converted"].sum())
print(
    f"Observational recurrence: {format_pt_br(total_rec)} purchases with another conversion within "
    f"{cfg.recurrence_window_days} days out of {format_pt_br(total_conv)} converted "
    f"({total_rec / total_conv:.1%}). Does not measure causal effect of the offer."
)

,offer_type,received,viewed,converted,recorrentes,revenue,cost,taxa_view,taxa_conversao,taxa_conversao_viewed,taxa_recorrencia,taxa_recorrencia_converted,margem_por_envio
0,bogo,30499,24514,13496,4375,270437.44,96795.0,0.8038,0.4425,0.5505,0.1434,0.3242,5.6934
1,discount,30543,20262,12500,3363,303389.07,34773.0,0.6634,0.4093,0.6169,0.1101,0.2690,8.7947
2,informational,15235,9878,8008,2141,89069.99,0.0,0.6484,0.5256,0.8107,0.1405,0.2674,5.8464


,campaign_wave,received,viewed,converted,taxa_view,taxa_conversao,taxa_conversao_viewed
0,0,12650,9718,5725,0.7682,0.4526,0.5891
1,1,12669,9634,6077,0.7604,0.4797,0.6308
2,2,12711,8915,6629,0.7014,0.5215,0.7436
3,3,12778,8904,5429,0.6968,0.4249,0.6097
4,4,12704,8523,5645,0.6709,0.4443,0.6623
5,5,12765,8960,4499,0.7019,0.3524,0.5021


,campaign_wave,received,converted,recorrentes,taxa_recorrencia,taxa_recorrencia_converted
0,0,12650,5725,1402,0.1108,0.2449
1,1,12669,6077,1755,0.1385,0.2888
2,2,12711,6629,2841,0.2235,0.4286
3,3,12778,5429,2420,0.1894,0.4458
4,4,12704,5645,1354,0.1066,0.2399
5,5,12765,4499,107,0.0084,0.0238


,offer_type,treatment,received,converted,recorrentes,taxa_recorrencia,taxa_recorrencia_converted
0,bogo,0,5985,2268,717,0.1198,0.3161
1,bogo,1,24514,11228,3658,0.1492,0.3258
2,discount,0,10281,2532,903,0.0878,0.3566
3,discount,1,20262,9968,2460,0.1214,0.2468
4,informational,0,5357,2340,589,0.1099,0.2517
5,informational,1,9878,5668,1552,0.1571,0.2738


Observational recurrence: 9.879 purchases with another conversion within 7 days out of 34.004 converted (29.1%). Does not measure causal effect of the offer.


## 9. Treatment, control, and spontaneous conversion

Source: processed. Compares those who viewed the offer with those who did not to assess whether the observed response should be treated as an uplift problem.

In [9]:
from pyspark.sql import functions as F

import numpy as np

from src import eda
from src.viz import format_pt_br, grouped_bars, markers_with_thresholds

# SPARK REDUCE
comparacao = eda.treatment_group_comparison(processed)
balanco = eda.covariate_balance(processed, cfg)

por_tipo = (
    processed.groupBy("treatment", "offer_type")
    .agg(
        F.count("*").alias("received"),
        F.sum("converted").alias("converted"),
        F.sum("conversion_value").alias("revenue"),
        F.sum("reward_cost").alias("cost"),
    )
    .toPandas()
    .assign(taxa_conversao=lambda d: d["converted"] / d["received"])
)

tabela_2x2 = (
    processed.groupBy("treatment", "converted")
    .agg(F.count("*").alias("linhas"))
    .toPandas()
)
pivot = (
    tabela_2x2.pivot(index="treatment", columns="converted", values="linhas")
    .fillna(0)
    .astype(int)
    .rename(columns={0: "did not convert", 1: "converted"})
)
pivot.index = pivot.index.map({0: "did not view (control)", 1: "viewed (treated)"})

taxa_controle = pivot.loc["did not view (control)", "converted"] / pivot.loc["did not view (control)"].sum()
taxa_tratado = pivot.loc["viewed (treated)", "converted"] / pivot.loc["viewed (treated)"].sum()

# PANDAS ANALYSIS
display(comparacao.round(4))
display(pivot)
display(por_tipo.round(4))
display(balanco.round(3))

por_tipo_wide = (
    por_tipo.pivot(index="offer_type", columns="treatment", values="taxa_conversao")
    .rename(columns={0: "did not view", 1: "viewed"})
    .reset_index()
)
grouped_bars(
    por_tipo_wide,
    category="offer_type",
    series=["did not view", "viewed"],
    series_names=["did not view (control)", "viewed (treated)"],
    title="Viewers convert more — but viewing is the costmer's choice",
    subtitle="taxa_conversao = converted / received, by arm and type",
    ylabel="taxa",
    orientation="v",
    value_label="%{y:.1%}",
).show()

desbalanceadas = balanco.loc[balanco["acima_do_limiar"], "covariavel"].tolist()
n_acima = len(desbalanceadas)

comparavel = balanco[np.isfinite(balanco["smd"])].copy()
comparavel["control_index"] = 100.0
comparavel["treated_index"] = np.where(
    comparavel["media_controle"].abs() > 1e-9,
    100 * comparavel["media_tratado"] / comparavel["media_controle"],
    np.where(comparavel["media_tratado"].abs() < 1e-9, 100.0, np.nan),
)
comparavel = comparavel.dropna(subset=["treated_index"]).sort_values("abs_smd")

title_index = (
    f"With hist_*, {n_acima} covariates separate viewers from non-viewers"
    if n_acima
    else "Profile and hist_* nearly equal across arms"
)
grouped_bars(
    comparavel,
    category="covariavel",
    series=["control_index", "treated_index"],
    series_names=["did not view (control)", "viewed (treated)"],
    title=title_index,
    subtitle=(
        f"index with control = 100 · includes {len(eda.HIST_BALANCE_FEATURES)} hist_* features · "
        f"limiar |SMD| = {cfg.smd_threshold}"
    ),
    orientation="h",
    value_label="%{x:.0f}",
    xlabel="index (control = 100)",
).show()

title_balance = (
    f"Viewing the offer is not random: {n_acima} covariate(s) above |SMD| {cfg.smd_threshold}"
    if n_acima
    else f"Treated and control are comparable: no covariate above |SMD| {cfg.smd_threshold}"
)

markers_with_thresholds(
    comparavel,
    value="smd",
    label="covariavel",
    flag="acima_do_limiar",
    thresholds=(-cfg.smd_threshold, cfg.smd_threshold),
    title=title_balance,
    subtitle=(
        "SMD on profile + hist_* + gender — those who open the offer already had more history "
        "of offers and purchases; diagnostic, not a gate"
    ),
    xlabel=f"SMD (linhas pontilhadas: ±{cfg.smd_threshold})",
).show()

print(
    f"Conversion in control (did not view): {taxa_controle:.1%}; in treated (viewed): {taxa_tratado:.1%}. "
    "μ₀ > 0 makes uplift estimable — viewing is treatment, not label (G3)."
)
print(
    f"Covariates above SMD threshold={cfg.smd_threshold}: "
    f"{n_acima} ({', '.join(desbalanceadas) if desbalanceadas else 'none'}). "
    "Imbalance shows up in engagement hist_* — viewers already received/bought more; "
    "demographic profile alone still looks balanced."
)

,treatment,received,taxa_conversao,ticket_medio,revenue_media
0,did not view,21623,0.3302,19.6469,6.4875
1,viewed,54654,0.4915,19.4542,9.5623


converted,did not convert,converted
treatment,,
did not view (control),14483,7140
viewed (treated),27790,26864


,treatment,offer_type,received,converted,revenue,cost,taxa_conversao
0,1,bogo,24514,11228,227429.21,82505.0,0.4580
1,0,discount,10281,2532,71774.63,8421.0,0.2463
2,1,discount,20262,9968,231614.44,26352.0,0.4920
3,1,informational,9878,5668,63573.98,0.0,0.5738
4,0,informational,5357,2340,25496.01,0.0,0.4368
5,0,bogo,5985,2268,43008.23,14290.0,0.3789


,covariavel,media_tratado,media_controle,smd,abs_smd,acima_do_limiar
0,hist_offers_received,1.808,2.034,-0.156,0.156,True
1,hist_offers_received_discount,0.718,0.833,-0.127,0.127,True
2,hist_txn_count,3.331,3.774,-0.120,0.120,True
3,hist_offers_received_bogo,0.721,0.813,-0.104,0.104,True
4,hist_frequency,0.198,0.217,-0.097,0.097,False
5,hist_completed_unseen_flag,0.142,0.176,-0.092,0.092,False
6,identity_missing,0.136,0.109,0.082,0.082,False
7,gender=unknown,0.136,0.109,0.082,0.082,False
8,credit_card_limit,65770.918,64392.859,0.063,0.063,False
9,gender=M,0.492,0.521,-0.059,0.059,False


Conversion in control (did not view): 33.0%; in treated (viewed): 49.2%. μ₀ > 0 makes uplift estimable — viewing is treatment, not label (G3).
Covariates above SMD threshold=0.1: 4 (hist_offers_received, hist_offers_received_discount, hist_txn_count, hist_offers_received_bogo). Imbalance shows up in engagement hist_* — viewers already received/bought more; demographic profile alone still looks balanced.


## 10. Observed offer economics

Source: processed. Examines conversion value, reward cost, and ticket to connect statistical response with business impact; includes discount-over-margin by offer type for uplift breakeven.

In [10]:
from pyspark.sql import functions as F

import numpy as np

from src import eda
from src.viz import faceted, format_pt_br, grouped_bars

# SPARK REDUCE
economia = (
    processed.groupBy("offer_type", "treatment")
    .agg(
        F.count("*").alias("received"),
        F.sum("converted").alias("converted"),
        F.sum("conversion_value").alias("revenue"),
        F.sum("reward_cost").alias("cost"),
    )
    .orderBy("offer_type", "treatment")
    .toPandas()
)
economia["margem"] = economia["revenue"] - economia["cost"]
economia["margem_por_envio"] = economia["margem"] / economia["received"]
economia["ticket_medio"] = np.where(
    economia["converted"] > 0,
    economia["revenue"] / economia["converted"],
    np.nan,
)

por_wave = (
    processed.groupBy("campaign_wave", "offer_type")
    .agg(
        F.count("*").alias("received"),
        F.sum("conversion_value").alias("revenue"),
        F.sum("reward_cost").alias("cost"),
    )
    .toPandas()
    .assign(margem_por_envio=lambda d: (d["revenue"] - d["cost"]) / d["received"])
    .sort_values(["offer_type", "campaign_wave"])
)

# PANDAS ANALYSIS
display(economia.round(2))
display(por_wave.round(2))

tratado = economia[economia["treatment"] == 1].copy()
grouped_bars(
    tratado,
    category="offer_type",
    series=["revenue", "cost", "margem"],
    series_names=["revenue", "cost", "net margin"],
    title="Discount yields more margin per send than bogo in the exposed arm",
    subtitle="treatment=1 only (viewed) · values in R$",
    ylabel="R$",
    orientation="v",
).show()

grouped_bars(
    tratado,
    category="offer_type",
    series=["margem_por_envio"],
    series_names=["margem / envio"],
    title="Observed profit per send does not match conversion rate",
    subtitle="margem_por_envio = (revenue − cost) / received",
    ylabel="R$ por envio",
    orientation="v",
).show()

ratio_desconto = eda.discount_margin_ratio_by_offer_type(processed)
display(
    ratio_desconto[
        [
            "offer_type",
            "conversoes",
            "valor_medio",
            "custo_medio",
            "margem_media",
            "desconto_sobre_margem_media",
            "desconto_sobre_margem_mediana",
            "desconto_sobre_margem_ponderada",
        ]
    ].round(3)
)

grouped_bars(
    ratio_desconto,
    category="offer_type",
    series=["desconto_sobre_margem_media", "desconto_sobre_margem_ponderada"],
    series_names=["média por conversão", "ponderada (Σcost / Σmargin)"],
    title="Bogo needs more uplift per real of margin than discount to break even",
    subtitle="converted=1 · reward_cost / (conversion_value − reward_cost) · linha em 1 = desconto = margem",
    ylabel="desconto / margem",
    orientation="v",
    value_label="%{y:.2f}",
).show()

painel_desconto = []
for tipo in ("bogo", "discount"):
    margem = F.col("conversion_value") - F.col("reward_cost")
    subset = (
        processed.filter(
            (F.col("converted") == 1) & (F.col("offer_type") == tipo) & (margem > 0)
        )
        .withColumn("desconto_sobre_margem", F.col("reward_cost") / margem)
    )
    hist = eda.numeric_histogram(subset, "desconto_sobre_margem", cfg.histogram_bins)
    painel_desconto.append({"title": tipo, "x": hist["centro"], "y": hist["contagem"]})

faceted(
    painel_desconto,
    title="Discount-over-margin spreads within each offer type",
    subtitle=f"converted=1, margem > 0 · n={format_pt_br(int(ratio_desconto.loc[ratio_desconto['offer_type'] != 'informational', 'conversoes'].sum()))} paid conversions",
    xlabel="reward_cost / (conversion_value − reward_cost)",
    vline=1.0,
    y_tickformat=",",
).show()

print(
    "High conversion and high value per send do not point to the same offer: "
    "bogo converts more, discount yields more per send in the treated arm."
)

,offer_type,treatment,received,converted,revenue,cost,margem,margem_por_envio,ticket_medio
0,bogo,0,5985,2268,43008.23,14290.0,28718.23,4.80,18.96
1,bogo,1,24514,11228,227429.21,82505.0,144924.21,5.91,20.26
2,discount,0,10281,2532,71774.63,8421.0,63353.63,6.16,28.35
3,discount,1,20262,9968,231614.44,26352.0,205262.44,10.13,23.24
4,informational,0,5357,2340,25496.01,0.0,25496.01,4.76,10.90
5,informational,1,9878,5668,63573.98,0.0,63573.98,6.44,11.22


,campaign_wave,offer_type,received,revenue,cost,margem_por_envio
17,0,bogo,5018,47463.84,16520.0,6.17
3,1,bogo,5118,50218.66,17910.0,6.31
2,2,bogo,5047,53834.34,18815.0,6.94
13,3,bogo,5110,39079.56,14755.0,4.76
6,4,bogo,5124,43098.07,15995.0,5.29
14,5,bogo,5082,36742.97,12800.0,4.71
4,0,discount,5093,51812.90,6015.0,8.99
8,1,discount,5015,54269.27,6114.0,9.60
10,2,discount,5129,61874.56,6990.0,10.70
1,3,discount,5100,50198.28,5815.0,8.70


High conversion and high value per send do not point to the same offer: bogo converts more, discount yields more per send in the treated arm.


## 11. Synthesis for modeling

Source: raw and processed. Summarizes findings that should guide modeling of which offer to send to each customer.

In [11]:
from src.viz import format_pt_br

# PANDAS ANALYSIS — synthesis of findings guiding uplift modeling

sintese = pd.DataFrame([
    {
        "finding": "Condensed grain and discrete sends",
        "evidence": f"{format_pt_br(events.count())} events → {format_pt_br(processed.count())} receipts across {len(ondas)} waves",
        "modeling_implication": "Features as-of received_time; campaign_wave captures timing, not continuous",
        "business_implication": "Budget and validity follow waves, not a continuous calendar",
    },
    {
        "finding": "Control converts without viewing (G3)",
        "evidence": f"control rate {taxa_controle:.1%} vs treated {taxa_tratado:.1%}; completions without view {taxa_sem_view:.1%}",
        "modeling_implication": "X-learner with μ₀ > 0; label ≠ exposure",
        "business_implication": "Offer moves those who would buy anyway and those who need a nudge",
    },
    {
        "finding": "bogo/discount/informational mix",
        "evidence": "10 offers · funnel and margin differ by type",
        "modeling_implication": "Stratify by offer_type; informational excluded from modeling",
        "business_implication": "Decision is which coupon to send, not just whether to send",
    },
    {
        "finding": "Informative historical features",
        "evidence": "hist_spend_total, hist_view_rate and hist_time_view_to_conv with tail and correlation",
        "modeling_implication": "hist_* enter the X-learner; recency and rates capture pre-send habit",
        "business_implication": "Those who already buy/view differently respond differently to the offer",
    },
    {
        "finding": "Economia ≠ conversion rate",
        "evidence": "discount: higher margin/send; bogo: higher conversion",
        "modeling_implication": "Score de ranking precisa ponderar τ e ticket/cost",
        "business_implication": "Optimize net profit, not just conversion",
    },
    {
        "finding": "Censoring and post-view imbalance",
        "evidence": (
            f"late waves with lower conversion; "
            f"{len(desbalanceadas)} hist_* acima de |SMD| {cfg.smd_threshold} "
            f"({', '.join(desbalanceadas[:3])}{'…' if len(desbalanceadas) > 3 else ''})"
        ),
        "modeling_implication": (
            "Temporal holdout; uplift under conditional ignorability on features — "
            "not by randomization of viewing"
        ),
        "business_implication": "Effect of late waves is underestimated; viewers were already more engaged",
    },
])

display(sintese)

print(
    "Modeling question: for each costmer, which offer (bogo or discount) "
    "maximizes net incremental profit — conversion τ minus reward cost?"
)

,finding,evidence,modeling_implication,business_implication
0,Condensed grain and discrete sends,306.534 events → 76.277 receipts across 6 waves,Features as-of received_time; campaign_wave ca...,"Budget and validity follow waves, not a contin..."
1,Control converts without viewing (G3),control rate 33.0% vs treated 49.2%; completio...,X-learner with μ₀ > 0; label ≠ exposure,Offer moves those who would buy anyway and tho...
2,bogo/discount/informational mix,10 offers · funnel and margin differ by type,Stratify by offer_type; informational excluded...,"Decision is which coupon to send, not just whe..."
3,Informative historical features,"hist_spend_total, hist_view_rate and hist_time...",hist_* enter the X-learner; recency and rates ...,Those who already buy/view differently respond...
4,Economia ≠ conversion rate,discount: higher margin/send; bogo: higher con...,Score de ranking precisa ponderar τ e ticket/cost,"Optimize net profit, not just conversion"
5,Censoring and post-view imbalance,late waves with lower conversion; 4 hist_* aci...,Temporal holdout; uplift under conditional ign...,Effect of late waves is underestimated; viewer...


Modeling question: for each costmer, which offer (bogo or discount) maximizes net incremental profit — conversion τ minus reward cost?


## Teardown

Stops the Spark session.

In [12]:
# SPARK TEARDOWN
spark.stop()
print("Spark session stopped.")

Spark session stopped.
